# Drishti-PAM — Cloud reference run (Kaggle)

This notebook lifts every Hugging Face code path from the local **mock** build into a real run on Kaggle. The free **GPU T4 x2** accelerator finishes the entire pipeline in ~12–15 minutes. It produces the same artefacts the local pipeline does (`sessions.json`, `labels.json`, `split.json`, `llm_model/`, `llm_predictions.json`, `comparison.json`) plus a downloadable `artifacts.zip`.

**What changes vs the local mock build?**

| Component | Local (mock) | Kaggle (this notebook) |
|---|---|---|
| `generator/llm_augment.py` justification paraphraser | deterministic synonym swap | **Flan-T5-base** real paraphrase via `AutoModelForSeq2SeqLM` |
| `generator/llm_augment.py` command-pool expander | inline-comment / echo wrappers | **Flan-T5-base** purpose-phrase wrap (preserves original verbatim) |
| `llm_detector/train.py` classifier | log-odds Naive Bayes (~0.81 F1) | fine-tuned **HybridDistilBERT**: DistilBERT [CLS] + cat-feature MLP, bf16 mixed precision on T4 |
| `llm_detector/detect.py` inference | weighted feature sum | hybrid forward pass (text + 19-dim cat vector) on GPU |

Every other file is unchanged.

**Notebook prerequisites (set in Notebook Settings):**
1. Accelerator → **GPU T4 x2** (Step 1 below hard-fails otherwise so you don't waste a 12-min CPU run).
2. Internet → **On** (needed once, to fetch model weights from Hugging Face).

## Step 0 — Verify GPU is actually attached

Kaggle silently falls back to CPU if the accelerator dropdown wasn't switched. This cell asserts CUDA is live so the rest of the run can't proceed in a degraded state.

In [ ]:
import torch

assert torch.cuda.is_available(), (
    'GPU not detected. Open Notebook Settings (right-hand pane) and set '
    'Accelerator to "GPU T4 x2", then restart the runtime and re-run this cell.'
)
n = torch.cuda.device_count()
for i in range(n):
    p = torch.cuda.get_device_properties(i)
    print(f'  GPU {i}: {p.name}, {p.total_memory / 1e9:.1f} GB, '
          f'compute {p.major}.{p.minor}')
print(f'  torch={torch.__version__}, '
      f'bf16 supported={torch.cuda.is_bf16_supported()}')

## Step 0.5 — Hugging Face token (optional but strongly recommended)

Anonymous downloads from Hugging Face are rate-limited (~10 GB / hour for the whole notebook IP). On a fresh Kaggle session we pull ~1.2 GB of weights (Flan-T5-base 990 MB + DistilBERT 250 MB), which fits under the limit when the IP is fresh — but if Kaggle has been heavily used by other notebooks on the same IP, downloads stall mid-run or fail with HTTP 429. The fine-tune then can't load the base model and the whole notebook errors out near minute 7.

**The 30-second fix:**

1. Visit [huggingface.co/settings/tokens](https://huggingface.co/settings/tokens) and create a **Read** token.
2. In Kaggle: right-pane → **Add-ons** → **Secrets** → **Add a new secret** → name `HF_TOKEN`, paste the token value, save.
3. Run the next cell. Authenticated downloads are ~2× faster and bypass the anonymous rate limit entirely.

If you don't have a token yet, the next cell logs an advisory and the run continues anonymously — no failure, just slower and more brittle.

In [ ]:
import os

try:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    HF_TOKEN = user_secrets.get_secret('HF_TOKEN')
    os.environ['HF_TOKEN'] = HF_TOKEN
    os.environ['HUGGINGFACE_HUB_TOKEN'] = HF_TOKEN
    masked = (HF_TOKEN[:6] + '...' + HF_TOKEN[-4:]
              if len(HF_TOKEN) > 12 else '***')
    print(f'  HF_TOKEN loaded from Kaggle Secrets: {masked}')
    print(f'  Forwarded to all child processes via os.environ.')
    print(f'  Downloads will use authenticated endpoints (no rate limit).')
except Exception as e:
    print(f'  No HF_TOKEN in Kaggle Secrets — anonymous downloads only.')
    print(f'  Reason: {type(e).__name__}: {e}')
    print(f'')
    print(f'  To add (one-time, takes 30 seconds):')
    print(f'    1. https://huggingface.co/settings/tokens -> create Read token')
    print(f'    2. Add-ons (right pane) -> Secrets -> HF_TOKEN -> save')
    print(f'    3. Re-run this cell')
    print(f'')
    print(f'  Continuing anonymously. If pip install or model download hits')
    print(f'  HTTP 429 / `_HFRequiredAuthError`, restart the runtime AFTER')
    print(f'  adding the secret.')

## Step 1 — Install / upgrade dependencies

Kaggle ships transformers and torch by default but the resident transformers is usually months behind the latest stable. We force an upgrade so we always get a v5+ build with the correct `processing_class` Trainer API and the `AutoModelForSeq2SeqLM` flow used by the generator. The `--upgrade` flag here does not touch torch (which has the right CUDA build for the Kaggle GPU).

In [ ]:
!pip install -q --upgrade "transformers>=5.5,<6" "datasets>=3" "accelerate>=1" "huggingface_hub>=1.0" faker matplotlib
!pip show transformers torch datasets accelerate | grep -E 'Name|Version'

## Step 2 — Pull the project source

Two options: clone from your own fork, or upload the project folder as a Kaggle Dataset and copy it into the working directory. The second option keeps Kaggle off the public internet which is preferable for corporate use.

In [ ]:
# Option A: clone from your own GitHub fork.
# !git clone https://github.com/YOUR-USER/YOUR-REPO.git /kaggle/working/code

# Option B (recommended): upload as a Kaggle Dataset, then copy it.
import shutil, os
SRC = '/kaggle/input/pam-source'             # adjust to your dataset slug
DST = '/kaggle/working/code'
if not os.path.isdir(DST):
    shutil.copytree(SRC, DST)
os.chdir(DST)
!ls -la

## Step 2.5 — Run parameters (edit me)

Tweak these before running Step 3. They drive the `smoke.py` invocation in the next cell.

| Parameter | Default | Lower bound | What it controls |
|---|---|---|---|
| `NUM_SESSIONS` | 10000 | 50 | Total synthetic sessions to emit. The 8 000 / 1 000 / 1 000 train/val/test split is computed from this. Lower numbers are useful for quick iteration on Kaggle's 30 GPU-hours/week budget — 1 000 sessions finishes the whole pipeline in ~3 min on T4. |
| `EXPAND_FACTOR` | 2 | 0 | Number of HF-generated variants per command in the pool expansion stage. 1 is the local-mock baseline; 2 is the published reference run; 0 disables command-pool expansion entirely. Each variant adds ~1 batched GPU call per source command, so the cost scales linearly. |
| `BATCH_SIZE` | 16 | 1 | `PAM_HF_BATCH_SIZE` — Flan-T5 batched-generation batch size. T4 (16 GB): leave at 16. A100 / H100 / RTX 4090 (24 GB+): bump to 64 to roughly halve Stage 1. 8 GB GPU OOM: drop to 4 or 8. |
| `MAX_NEW_TOKENS` | 80 | 32 | `PAM_HF_MAX_NEW_TOKENS` — upper bound on per-paraphrase generation length. 80 is generous for a justification (~25 T5 tokens); raising it costs you proportionally more wall-clock per batch. |
| `TEMPERATURE` | 0.8 | 0.1 | `PAM_HF_TEMPERATURE` — softmax temperature for nucleus sampling. 0.8 is the "creative-but-coherent" sweet spot. **Greedy decoding (which would be temperature ≈ 0) collapses Flan-T5 onto a verbatim-copy attractor for IT change-management text — kept-rate falls to ~30 % even on Flan-T5-large.** Raise to 0.9-1.0 for more diversity (slightly higher hallucination risk on numbers / asset names); lower to 0.6 for more conservative paraphrases. To revert to greedy entirely (only useful for ablations) set `os.environ['PAM_HF_SAMPLE'] = 'false'`. |

In [ ]:
import os

# NUM_SESSIONS = 1_000  # smoke test only — Stage 5 F1 is UNRELIABLE on
#                         this size: rare scenarios (S14 privilege
#                         escalation, S15 approver-as-implementer) are
#                         likely absent from the 100-session test split,
#                         and the DistilBERT fine-tune runs only ~75
#                         optimizer steps. Use 10_000 for any number
#                         you want to publish or compare against the
#                         published reference numbers.
NUM_SESSIONS    = 10_000   # full reference run
EXPAND_FACTOR   = 2        # HF command-pool variants per command
BATCH_SIZE      = 16       # Flan-T5 batched-generation batch size
MAX_NEW_TOKENS  = 80       # cap on per-paraphrase generation length
TEMPERATURE     = 0.8      # nucleus-sampling temperature (greedy ≈ 0)

# Optional advanced overrides — uncomment to tune.
# os.environ['PAM_HF_PARAPHRASE_MODEL'] = 'google/flan-t5-large'  # 770M instead of 250M
# os.environ['PAM_HF_TOP_P']            = '0.9'                    # nucleus cut-off
# os.environ['PAM_HF_SEED']             = '42'                     # sampling RNG seed
# os.environ['PAM_HF_SAMPLE']           = 'false'                  # ablation: greedy decoding

# All of these propagate to every child process via os.environ.
os.environ['PAM_HF_BATCH_SIZE']     = str(BATCH_SIZE)
os.environ['PAM_HF_MAX_NEW_TOKENS'] = str(MAX_NEW_TOKENS)
os.environ['PAM_HF_TEMPERATURE']    = str(TEMPERATURE)

print(f'Run parameters:')
print(f'  NUM_SESSIONS    = {NUM_SESSIONS:,}')
print(f'  EXPAND_FACTOR   = {EXPAND_FACTOR}')
print(f'  BATCH_SIZE      = {BATCH_SIZE}  (PAM_HF_BATCH_SIZE)')
print(f'  MAX_NEW_TOKENS  = {MAX_NEW_TOKENS}  (PAM_HF_MAX_NEW_TOKENS)')
print(f'  TEMPERATURE     = {TEMPERATURE}  (PAM_HF_TEMPERATURE)')
print(f'  paraphrase model = {os.environ.get("PAM_HF_PARAPHRASE_MODEL", "google/flan-t5-base")}')
print(f'  decode sampling  = {os.environ.get("PAM_HF_SAMPLE", "true")}')

## Step 3 — Run the entire pipeline with the HuggingFace backend

`smoke.py --backend huggingface --expand-factor 2` orchestrates every stage in order:

1. **Generate** the dataset — Flan-T5-base (loaded via `AutoModelForSeq2SeqLM`, fp16 on GPU) paraphrases every category-template justification, and every command pool is expanded with two HF-generated semantic variants per command. The HF expansion wraps the original command verbatim (`echo '<purpose phrase>' && <cmd>`) so destructive / sensitive / lateral / escalation tokens always survive in the augmented pool.
2. **Score the rule detector** on all 10 000 sessions (no LLM dependency).
3. **Fine-tune DistilBERT** for 3 epochs on the 8 000-session train partition with bf16 mixed precision (T4 supports bf16). Saves to `data/llm_model/`. ~5–6 minutes on T4 x2.
4. **Score the test split** with the fine-tuned DistilBERT on GPU.
5. **Compare** rule vs DistilBERT on the test split. Expect DistilBERT to outperform the mock log-odds classifier on the augmented data — Flan-T5 paraphrases break the simple token signals the mock relies on, but DistilBERT's contextual embedding is robust to surface variation.

Once Stage 5 finishes, the headline numbers are written to `data/comparison.json`. Run `python docs/build_dashboard.py` from `code/` to refresh the dashboard with the new run, or download `artifacts.zip` (Step 4 below) and unpack it locally.

In [ ]:
# First call downloads google/flan-t5-base (~990 MB) and distilbert-base-uncased
# (~250 MB) into Kaggle's HF cache. Subsequent runs reuse them.
#
# NOTE: PAM_HF_BATCH_SIZE and PAM_HF_MAX_NEW_TOKENS were exported from the
# parameters cell above and are inherited by the child process here.
!python smoke.py --backend huggingface \
                 --expand-factor {EXPAND_FACTOR} \
                 --num-sessions {NUM_SESSIONS} \
                 --non-interactive

### Quick iteration: lower the session count

If you're iterating on the trainer or the data and don't need 10k sessions every time:

```bash
!python smoke.py --backend huggingface \
                 --expand-factor {EXPAND_FACTOR} \
                 --num-sessions 1000 \
                 --non-interactive
```

For a 60-second sanity loop on Kaggle: set `NUM_SESSIONS = 1000` in the parameters cell. The full pipeline finishes in ~3 minutes on T4 — you still get real Flan-T5 paraphrasing and a real DistilBERT fine-tune, just on a 700/150/150 split instead of 8 000/1 000/1 000.

Or run individual stages by hand — every script accepts `--help` for its full flag surface.

### Falling back to mock for a sanity check

If the cloud run hits an unexpected error, run the same harness with the offline backend to confirm the rest of the pipeline is healthy:

```bash
!python smoke.py --backend mock --non-interactive  # ~10 s, no Hugging Face dependency
```

## Step 4 — Bundle artefacts for download

Zips up everything that's worth pulling back to your laptop:

  * `data/sessions.json` and `data/labels.json` (cloud-augmented dataset)
  * `data/split.json`
  * `data/rule_predictions.json`, `data/llm_predictions.json`
  * `data/comparison.json`, `data/comparison.md`
  * `data/llm_model/` (the fine-tuned DistilBERT, ~250 MB)

Once the cell completes, the file appears in Kaggle's right-hand pane under `/kaggle/working/`.

In [ ]:
import shutil, os
shutil.make_archive('/kaggle/working/artifacts', 'zip', root_dir='data')
size_mb = os.path.getsize('/kaggle/working/artifacts.zip') / (1024 * 1024)
print(f'artifacts.zip = {size_mb:.1f} MB')
!ls -la /kaggle/working/artifacts.zip

## Bringing the artefacts back home

1. Download `artifacts.zip` from the Kaggle right-hand pane.
2. Unzip it into `code/data/` on your local machine.
3. (Optional) Replace `data/llm_model.json` (mock pointer) with `data/llm_model/` (the real model directory).
4. Run `python compare.py` locally to reproduce the cloud comparison without any HF dependency.
5. Run `python docs/build_dashboard.py` to refresh the dashboard with the new headline numbers, then `python -m http.server 8000 --directory docs` to view it locally.